# Semaine 3 — Jour 7 — Projet multi-agent

Notebook étudiant généré depuis les fichiers Markdown du jour.

# Objectifs pédagogiques — Jour 7 — Projet multi-agent

## Objectifs principaux

À la fin de cette journée, l’apprenant doit savoir :

1. concevoir un projet multi-agent complet ;
2. décomposer une demande utilisateur en tâches distribuables ;
3. choisir les agents nécessaires à l’exécution d’un objectif ;
4. distinguer orchestration, handoff, outil et mémoire ;
5. exposer des capacités via une couche compatible MCP ;
6. maintenir un état partagé sans fuite de contexte privé ;
7. construire un contexte ciblé pour chaque agent ;
8. tracer chaque étape d’exécution ;
9. intégrer une revue qualité avant réponse finale ;
10. expliquer les limites d’un système multi-agent autonome.

## Objectifs d’architecture

L’apprenant doit être capable de justifier :

- pourquoi un agent central coordonne le workflow ;
- pourquoi tous les agents ne reçoivent pas tout le contexte ;
- pourquoi l’état partagé est versionné ;
- pourquoi les outils sensibles nécessitent une approbation ;
- pourquoi un reviewer indépendant est utile ;
- pourquoi le projet doit avoir des critères d’arrêt explicites.

## Objectifs de code

Le lab doit permettre de pratiquer :

- `dataclasses` pour représenter état, tâches, artefacts et événements ;
- validation d’entrée sans dépendance externe ;
- registre d’outils déterministe ;
- simulation d’appels MCP ;
- filtrage de contexte par rôle ;
- orchestration séquentielle contrôlée ;
- export de trace JSON ;
- tests unitaires avec `unittest`.

## Ce que cette journée ne couvre pas

Cette journée ne couvre pas encore :

- le déploiement production ;
- la persistance en base de données ;
- les files de messages distribuées ;
- l’observabilité complète ;
- la gestion de coûts en production ;
- le sandboxing système avancé.

Ces sujets seront approfondis dans les semaines suivantes.


# Chapitre — Projet multi-agent

## 1. Pourquoi un projet intégrateur ?

Les premiers jours de la semaine ont introduit les briques de base :

- une architecture multi-agents ;
- une stratégie de coordination ;
- un serveur MCP ;
- un client MCP ;
- un état partagé ;
- une politique de contexte.

Un système réel ne se contente pas d’empiler ces composants. Il doit les organiser autour d’un objectif métier.

Le projet du jour répond à une question simple :

> Comment construire un système multi-agent qui produit un résultat utile, contrôlé et vérifiable ?

## 2. Le piège du “plus d’agents”

Une erreur fréquente consiste à ajouter des agents pour chaque sous-problème.

Ce n’est pas une architecture. C’est une fragmentation.

Un bon système multi-agent commence par identifier les responsabilités stables :

| Responsabilité | Agent possible |
|---|---|
| Décomposer le travail | Planner |
| Chercher ou extraire le contexte | Researcher |
| Produire une solution technique | Engineer |
| Vérifier les risques | Security |
| Contrôler la qualité | Reviewer |
| Coordonner le tout | Coordinator |

Un agent doit exister seulement si sa responsabilité est claire et si sa sortie est utile à un autre composant.

## 3. Architecture cible

Le projet du jour utilise une architecture **coordinator-led**.

Le coordinateur ne fait pas tout lui-même. Il :

1. reçoit l’objectif ;
2. valide la demande ;
3. crée un plan ;
4. sélectionne les agents ;
5. construit un contexte adapté à chaque agent ;
6. appelle les outils nécessaires ;
7. stocke les résultats dans l’état partagé ;
8. déclenche une revue ;
9. décide si le résultat est acceptable.

Cette architecture est plus simple à auditer qu’un réseau d’agents qui se délèguent librement des responsabilités.

## 4. MCP dans le projet

MCP est utilisé ici comme modèle d’intégration d’outils.

Dans un projet réel, un client MCP peut découvrir les capacités disponibles sur un serveur, puis appeler des tools, lire des resources ou utiliser des prompts.

Dans le lab, l’objectif est pédagogique :

- le serveur MCP est simulé par un registre local ;
- les appels respectent une forme standardisée ;
- chaque outil déclare un schéma d’entrée ;
- le client valide les arguments avant exécution ;
- les outils sensibles sont protégés par approbation humaine.

Cette approche rend le design portable vers un vrai serveur MCP plus tard.

## 5. État partagé

Un projet multi-agent a besoin d’un état partagé, mais cet état ne doit pas devenir un dépôt global incontrôlé.

Le lab distingue trois niveaux de visibilité :

- `private` : visible uniquement par l’agent propriétaire ;
- `shared` : visible par les agents autorisés ;
- `public` : visible par tous les agents du workflow.

Cette séparation limite les fuites de contexte, réduit le bruit et facilite l’audit.

## 6. Context Engineering

Chaque agent reçoit un **context pack** spécifique.

Le contexte n’est pas l’historique complet. C’est une sélection contrôlée de ce qui est utile pour la tâche courante.

Un bon context pack contient :

- l’objectif ;
- la tâche courante ;
- les éléments d’état pertinents ;
- les artefacts disponibles ;
- les contraintes ;
- les outils autorisés ;
- les critères de sortie.

Le lab impose un budget de contexte approximatif en nombre de caractères. Ce budget force à prioriser.

## 7. Boucle de livraison

La boucle de livraison du projet suit le cycle suivant :

```text
objective
  -> plan
  -> assign
  -> build context
  -> call agent
  -> call tools
  -> update shared state
  -> review
  -> final answer
```

La boucle n’est pas infinie. Elle possède :

- une limite d’étapes ;
- des statuts contrôlés ;
- une revue finale ;
- un score minimum ;
- une trace exportable.

## 8. Critères d’acceptation

Le projet est considéré comme réussi si :

- l’objectif utilisateur est validé ;
- un plan est généré ;
- au moins deux agents spécialisés contribuent ;
- les outils nécessaires sont appelés via l’adaptateur MCP ;
- l’état partagé contient les contributions importantes ;
- la revue finale est positive ;
- la trace explique ce qui s’est passé ;
- le résultat final est sérialisable en JSON.

## 9. Ce qui rend l’architecture robuste

Le système devient plus robuste grâce à plusieurs choix :

### Responsabilités séparées

Le planner ne valide pas la sécurité.  
L’engineer ne note pas sa propre production.  
Le reviewer ne modifie pas directement les artefacts.  
Le coordinator orchestre, mais ne remplace pas les spécialistes.

### Interfaces explicites

Les agents échangent des `Task`, `Artifact` et `StateEntry`.

Cela évite les échanges informels impossibles à tester.

### État contrôlé

Chaque entrée d’état possède :

- une clé ;
- une valeur ;
- un propriétaire ;
- une visibilité ;
- une version.

### Outils bornés

Chaque outil possède :

- un nom ;
- une description ;
- un schéma d’entrée ;
- une fonction déterministe ;
- une règle d’approbation si nécessaire.

## 10. Limites pédagogiques

Le lab ne contacte pas de vrai modèle de langage.

C’est volontaire.

Avant de connecter un LLM, il faut comprendre :

- où passe l’état ;
- comment le contexte est construit ;
- qui peut appeler quoi ;
- comment sont validées les entrées ;
- comment sont évalués les résultats ;
- comment déboguer une exécution.

Un LLM peut remplacer les fonctions déterministes du lab, mais il ne doit pas remplacer l’architecture.

## 11. Extension vers production

Pour passer vers un système réel, on pourrait remplacer :

| Composant pédagogique | Équivalent production |
|---|---|
| Fonctions déterministes | Appels LLM |
| Registre local | Serveur MCP réel |
| État en mémoire | PostgreSQL ou Redis |
| Trace JSON locale | Observabilité distribuée |
| Reviewer simple | Évaluations automatisées |
| Validation manuelle booléenne | Workflow human-in-the-loop |

## 12. Message clé

Un projet multi-agent professionnel n’est pas une conversation entre plusieurs personnages.

C’est un système logiciel orchestré, testé, observable et contraint.


## Lab

Le lab est situé dans `book/week03/day07/labs/`.

In [ ]:
from pathlib import Path
import sys

# À exécuter depuis la racine du dépôt.
LAB_PATH = Path.cwd() / "book" / "week03" / "day07" / "labs"
if str(LAB_PATH) not in sys.path:
    sys.path.insert(0, str(LAB_PATH))

from multi_agent_project import DeliveryCoordinator, summarize_result

coordinator = DeliveryCoordinator()
result = coordinator.run(
    "Concevoir une architecture multi-agent avec MCP, état partagé, contexte contrôlé, sécurité et revue finale."
)
print(summarize_result(result))

In [ ]:
# Afficher les artefacts produits
for artifact in result["artifacts"]:
    print(f"- {artifact['name']} ({artifact['kind']}) par {artifact['created_by']}")

In [ ]:
# Observer la trace
for event in result["trace"][:8]:
    print(event)

# Exercices — Jour 7 — Projet multi-agent

## Exercice 1 — Identifier les responsabilités

À partir du scénario suivant :

> Un utilisateur demande un plan de migration d’un assistant interne vers une architecture multi-agent avec MCP, état partagé et revue sécurité.

Identifiez les responsabilités qui doivent être séparées entre agents.

Répondez sous forme de tableau :

| Responsabilité | Agent | Sortie attendue |
|---|---|---|

## Exercice 2 — Définir un état partagé

Proposez cinq entrées d’état utiles pour le projet.

Pour chaque entrée, précisez :

- la clé ;
- le propriétaire ;
- la visibilité ;
- la raison de stockage.

## Exercice 3 — Construire un context pack

Pour un agent `security`, listez les informations qui doivent être incluses dans son contexte.

Listez aussi les informations qui ne doivent pas être incluses.

## Exercice 4 — Identifier les outils MCP

Proposez quatre outils que le système pourrait exposer via MCP.

Pour chaque outil, indiquez :

- nom ;
- description ;
- arguments requis ;
- sortie attendue ;
- sensibilité.

## Exercice 5 — Critères d’arrêt

Définissez les conditions qui doivent arrêter la boucle multi-agent.

Incluez au moins :

- une condition de succès ;
- une condition d’échec ;
- une condition de clarification utilisateur ;
- une condition de sécurité.

## Exercice 6 — Trace d’exécution

Écrivez un exemple de trace en JSON pour une exécution courte contenant :

- un événement de planification ;
- un appel d’outil ;
- une contribution d’agent ;
- une revue finale.

## Exercice 7 — Analyse de robustesse

Expliquez pourquoi il est risqué de laisser tous les agents accéder à tout l’état partagé.

Donnez deux exemples de bugs ou d’effets indésirables.


# Questions d’entretien — Jour 7 — Projet multi-agent

## Question 1

Pourquoi un projet multi-agent ne doit-il pas être conçu simplement comme une liste d’agents ?

## Question 2

Quelle est la différence entre un agent spécialisé et un outil ?

## Question 3

Quel est le rôle du coordinateur dans une architecture multi-agent ?

## Question 4

Pourquoi faut-il construire un contexte différent pour chaque agent ?

## Question 5

Comment MCP aide-t-il à intégrer des outils dans un système multi-agent ?

## Question 6

Pourquoi un état partagé doit-il être versionné ?

## Question 7

Quelle différence faites-vous entre `private`, `shared` et `public` dans un store d’état ?

## Question 8

Pourquoi faut-il une revue finale indépendante ?

## Question 9

Quels critères utiliseriez-vous pour décider qu’une boucle multi-agent doit s’arrêter ?

## Question 10

Quels signaux faut-il tracer pour déboguer un système multi-agent ?


# Challenge — Jour 7 — Projet multi-agent

## Objectif

Étendre le lab pour construire un **assistant multi-agent de préparation de livraison logicielle**.

L’assistant doit recevoir un objectif comme :

> Préparer une livraison d’API avec documentation, tests, analyse des risques et plan de rollback.

## Contraintes

Votre solution doit :

1. ajouter un agent `qa`;
2. ajouter un outil `generate_release_checklist`;
3. produire au moins quatre artefacts ;
4. refuser les actions sensibles sans approbation humaine ;
5. maintenir un état partagé versionné ;
6. produire une trace JSON ;
7. déclencher une revue finale ;
8. retourner un statut contrôlé parmi :
   - `completed`;
   - `needs_revision`;
   - `needs_clarification`;
   - `blocked_by_policy`.

## Livrables attendus

Vous devez fournir :

- le code modifié ;
- les tests associés ;
- un exemple de sortie JSON ;
- une courte justification d’architecture.

## Critères d’évaluation

La solution est réussie si :

- les responsabilités des agents sont séparées ;
- les outils sont validés avant exécution ;
- le contexte transmis à chaque agent est limité ;
- la revue finale ne dépend pas de l’agent producteur ;
- la sortie est traçable ;
- les tests couvrent les cas de succès et d’échec.


# Références — Jour 7 — Projet multi-agent

## Documentation officielle

- OpenAI Agents SDK — Agents  
  https://openai.github.io/openai-agents-python/agents/

- OpenAI Agents SDK — Model Context Protocol  
  https://openai.github.io/openai-agents-python/mcp/

- OpenAI Agents SDK — Context management  
  https://openai.github.io/openai-agents-python/context/

- OpenAI Agents SDK — Handoffs  
  https://openai.github.io/openai-agents-python/handoffs/

- Model Context Protocol — Specification overview  
  https://modelcontextprotocol.io/specification/2025-06-18/basic/index

- Model Context Protocol — Tools  
  https://modelcontextprotocol.io/specification/2025-06-18/server/tools

- Model Context Protocol — Resources  
  https://modelcontextprotocol.io/specification/2025-06-18/server/resources

## Concepts à revoir

- Orchestration multi-agents
- Handoffs
- Tool registry
- MCP client/server
- État partagé
- Context engineering
- Guardrails
- Tracing
- Critères d’arrêt
- Human-in-the-loop

## Lecture active

En lisant la documentation, cherchez à répondre aux questions suivantes :

1. Qu’est-ce qui est responsabilité du framework ?
2. Qu’est-ce qui reste responsabilité de l’AI Engineer ?
3. Où sont placés les outils ?
4. Où passe le contexte ?
5. Comment les actions sensibles sont-elles contrôlées ?
6. Comment déboguer une exécution multi-agent ?
